In [17]:
import re
import unicodedata
import pandas as pd
import numpy as np
pd.options.display.float_format = '{:,.2f}'.format

In [18]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
sys.path.insert(0, str(PROJECT_ROOT))

from src.transform import sanitize, features

In [35]:
FILE_PATH = '../data/processed/Ações Afirmativas na Política Nacional Aldir Blanc - Análise de Descumprimentos.xlsx'

In [36]:
df = pd.read_excel(FILE_PATH, sheet_name='Resultado Final')

In [37]:
# Exclui a coluna de parecer
df = df.drop(columns=['Conferência/Parecer'])
df = df.reset_index(drop=True)

In [42]:
import numpy as np

mask_sem_vagas = df['vagas_totais'].isna()

df.loc[mask_sem_vagas, 'valor_cotas_negras'] = (
    df.loc[mask_sem_vagas, 'perc_cotas_negras'] * df.loc[mask_sem_vagas, 'valor_total']
)

df.loc[mask_sem_vagas, 'valor_cotas_indigenas'] = (
    df.loc[mask_sem_vagas, 'perc_cotas_indigenas'] * df.loc[mask_sem_vagas, 'valor_total']
)

df.loc[mask_sem_vagas, 'valor_cotas_pcd'] = (
    df.loc[mask_sem_vagas, 'perc_cotas_pcd'] * df.loc[mask_sem_vagas, 'valor_total']
)

In [46]:
# Recalcula as flags
df = features.cria_flags_cotas(df)

In [47]:
# Existe apenas uma linha que possui valor vazio para 'tipo_ente'. Trata-se do Maranhão - aqui faço essa correção
df.loc[df['tipo_ente'].isna(), 'tipo_ente'] = 'ESTADO'

In [48]:
# Separa as duas tabelas de análise: Estados e Capitais
df_estados  = df.loc[df['tipo_ente'] == 'ESTADO']
df_capitais = df.loc[df['tipo_ente'] == 'CAPITAL']

In [49]:
df['ente_federativo'].unique()

array(['AMAZONAS', 'MATO GROSSO DO SUL', 'RONDÔNIA', 'ACRE',
       'PORTO VELHO', 'FLORIANÓPOLIS', 'MANAUS', 'RIO BRANCO',
       'RIO DE JANEIRO', 'TERESINA', 'MARANHÃO', 'MATO GROSSO', 'ALAGOAS',
       'RIO GRANDE DO SUL', 'JOÃO PESSOA', 'GOIÁS', 'SANTA CATARINA',
       'DISTRITO FEDERAL', 'AMAPÁ', 'CEARÁ', 'ESPÍRITO SANTO',
       'MINAS GERAIS', 'PARANÁ', 'PARAÍBA', 'PARÁ', 'PERNAMBUCO', 'PIAUÍ',
       'RIO GRANDE DO NORTE', 'RORAIMA', 'SERGIPE', 'SÃO PAULO',
       'TOCANTINS', 'ARACAJU', 'BELÉM', 'BELO HORIZONTE', 'BOA VISTA',
       'CAMPO GRANDE', 'CUIABÁ', 'CURITIBA', 'FORTALEZA', 'GOIÂNIA',
       'MACAPÁ', 'MACEIÓ', 'NATAL', 'PALMAS', 'PORTO ALEGRE',
       'SÃO LUIS', 'SÃO PAULO', 'VITÓRIA', 'RECIFE', 'BAHIA',
       'SALVADOR'], dtype=object)

In [50]:
df.columns

Index(['ente_federativo', 'nome_pdf_pk', 'perc_cotas_negras',
       'perc_cotas_indigenas', 'perc_cotas_pcd', 'vagas_totais', 'valor_total',
       'is_novo', 'tipo_ente', 'vagas_cotas_negras', 'vagas_cotas_indigenas',
       'vagas_cotas_pcd', 'valor_por_vaga', 'valor_cotas_negras',
       'valor_cotas_indigenas', 'valor_cotas_pcd', 'flag_cotas_negras',
       'flag_cotas_indigenas', 'flag_cotas_pcd', 'tipo_edital'],
      dtype='object')

In [51]:
df['tipo_ente'].value_counts(dropna=False)

tipo_ente
ESTADO     351
CAPITAL    147
Name: count, dtype: int64

In [52]:
print("Valor total Estados")
print(df.loc[df['tipo_ente'] == 'ESTADO', 'valor_total'].sum()) # 1.350.486.136,3
print("Valor total Capitais")
print(df.loc[df['tipo_ente'] == 'CAPITAL', 'valor_total'].sum()) # 269.745.442,81

Valor total Estados
1350486136.3
Valor total Capitais
269745442.81


In [53]:
# Define as máscaras para cotas

mask_cotas_indigenas    = df_estados['flag_cotas_indigenas'].eq(1)
mask_cotas_pcd          = df_estados['flag_cotas_pcd'].eq(1)

In [31]:
# Cotas pessoas negras - Geral
mask_cotas_negras   = df['flag_cotas_negras'].eq(1)
qtd_cotas_negras    = mask_cotas_negras.sum()
qtd_total           = len(df)
percentual          = qtd_cotas_negras / qtd_total * 100 # 85.75%
soma_valor          = df['valor_cotas_negras'].sum() # 302.939.910,84
soma_vaga           = df['vagas_cotas_negras'].sum() 

print("==========================================")
print("Estados")
print("==========================================")
print("% Cumprimento Cotas")
print(percentual)
print("\nValor total")
print(soma_valor)
print("\nNúmero de vagas")
print(soma_vaga)
print("==========================================")

Estados
% Cumprimento Cotas
85.14056224899599

Valor total
377613782.93589926

Número de vagas
7245.0


In [54]:
# Cotas pessoas negras - Geral
mask_cotas_negras   = df['flag_cotas_negras'].eq(1)
qtd_cotas_negras    = mask_cotas_negras.sum()
qtd_total           = len(df)
percentual          = qtd_cotas_negras / qtd_total * 100 # 85.75%
soma_valor          = df['valor_cotas_negras'].sum() # 302.939.910,84
soma_vaga           = df['vagas_cotas_negras'].sum() 

print("==========================================")
print("Estados")
print("==========================================")
print("% Cumprimento Cotas")
print(percentual)
print("\nValor total")
print(soma_valor)
print("\nNúmero de vagas")
print(soma_vaga)
print("==========================================")


# Cotas pessoas negras - Estados
mask_cotas_negras   = df_estados['flag_cotas_negras'].eq(1)
qtd_cotas_negras    = mask_cotas_negras.sum()
qtd_total           = len(df_estados)
percentual          = qtd_cotas_negras / qtd_total * 100 # 85.75%
soma_valor          = df_estados['valor_cotas_negras'].sum() # 302.939.910,84
soma_vaga           = df_estados['vagas_cotas_negras'].sum() 

print("==========================================")
print("Estados")
print("==========================================")
print("% Cumprimento Cotas Negras")
print(percentual)
print("\nValor total para pessoas negras")
print(soma_valor)
print("\nNúmero de vagas")
print(soma_vaga)
print("==========================================")
# Cotas pessoas negras - Capitais
mask_cotas_negras   = df_capitais['flag_cotas_negras'].eq(1)
qtd_cotas_negras    = mask_cotas_negras.sum()
qtd_total           = len(df_capitais)
percentual          = qtd_cotas_negras / qtd_total * 100 # 83.75%
soma_valor          = df_capitais['valor_cotas_negras'].sum() # 56.346.222,21
soma_vaga           = df_capitais['vagas_cotas_negras'].sum() 
print("Capitais")
print("==========================================")
print("% Cumprimento Cotas Negras")
print(percentual)
print("\nValor total para pessoas negras")
print(soma_valor)
print("\nNúmero de vagas")
print(soma_vaga)
print("==========================================")

Estados
% Cumprimento Cotas
85.14056224899599

Valor total
377613782.93589926

Número de vagas
7245.0
Estados
% Cumprimento Cotas Negras
85.75498575498575

Valor total para pessoas negras
316798462.3428941

Número de vagas
5649.0
Capitais
% Cumprimento Cotas Negras
83.6734693877551

Valor total para pessoas negras
60815320.59300518

Número de vagas
1596.0


In [55]:
# Cotas pessoas indigenas - Geral
mask_cotas_negras   = df['flag_cotas_indigenas'].eq(1)
qtd_cotas_negras    = mask_cotas_negras.sum()
qtd_total           = len(df)
percentual          = qtd_cotas_negras / qtd_total * 100 # 85.75%
soma_valor          = df['valor_cotas_indigenas'].sum() # 302.939.910,84
soma_vaga           = df['vagas_cotas_indigenas'].sum() 

print("==========================================")
print("Estados")
print("==========================================")
print("% Cumprimento Cotas")
print(percentual)
print("\nValor total")
print(soma_valor)
print("\nNúmero de vagas")
print(soma_vaga)
print("==========================================")

# Cotas pessoas indigenas - Estados
mask_cotas_indigenas   = df_estados['flag_cotas_indigenas'].eq(1)
qtd_cotas_indigenas    = mask_cotas_indigenas.sum()
qtd_total           = len(df_estados)
percentual          = qtd_cotas_indigenas / qtd_total * 100 # 86.60%
soma_valor          = df_estados['valor_cotas_indigenas'].sum() # 118.190.085,67
soma_vaga           = df_estados['vagas_cotas_indigenas'].sum() 
print("==========================================")
print("Estados")
print("==========================================")
print("% Cumprimento Cotas indigenas")
print(percentual)
print("\nValor total para pessoas indigenas")
print(soma_valor)
print("\nNúmero de vagas")
print(soma_vaga)
print("==========================================")
# Cotas pessoas indigenas - Capitais
mask_cotas_indigenas   = df_capitais['flag_cotas_indigenas'].eq(1)
qtd_cotas_indigenas    = mask_cotas_indigenas.sum()
qtd_total           = len(df_capitais)
percentual          = qtd_cotas_indigenas / qtd_total * 100 # 87.75%
soma_valor          = df_capitais['valor_cotas_indigenas'].sum() # 22513919.55
soma_vaga           = df_capitais['vagas_cotas_indigenas'].sum() 

print("Capitais")
print("==========================================")
print("% Cumprimento Cotas indigenas")
print(percentual)
print("\nValor total para pessoas indigenas")
print(soma_valor)
print("\nNúmero de vagas")
print(soma_vaga)
print("==========================================")


Estados
% Cumprimento Cotas
86.94779116465864

Valor total
147971260.5898807

Número de vagas
2906.0
Estados
% Cumprimento Cotas indigenas
86.6096866096866

Valor total para pessoas indigenas
123942641.57346568

Número de vagas
2256.0
Capitais
% Cumprimento Cotas indigenas
87.75510204081633

Valor total para pessoas indigenas
24028619.016415007

Número de vagas
650.0


In [56]:
# Cotas pessoas indigenas - Geral
mask_cotas_pcd   = df['flag_cotas_pcd'].eq(1)
qtd_cotas_pcd    = mask_cotas_pcd.sum()
qtd_total           = len(df)
percentual          = qtd_cotas_pcd / qtd_total * 100
soma_valor          = df['valor_cotas_pcd'].sum() 
soma_vaga           = df['vagas_cotas_pcd'].sum() 

print("==========================================")
print("Geral")
print("==========================================")
print("% Cumprimento Cotas")
print(percentual)
print("\nValor total")
print(soma_valor)
print("\nNúmero de vagas")
print(soma_vaga)
print("==========================================")

# Cotas pessoas pcd - Estados
mask_cotas_pcd      = df_estados['flag_cotas_pcd'].eq(1)
qtd_cotas_pcd       = mask_cotas_pcd.sum()
qtd_total           = len(df_estados)
percentual          = qtd_cotas_pcd / qtd_total * 100 # 88.60%
soma_valor          = df_estados['valor_cotas_pcd'].sum() # 65.286.995,52
soma_vaga           = df_estados['vagas_cotas_pcd'].sum() 
print("==========================================")
print("Estados")
print("==========================================")
print("% Cumprimento Cotas pcd")
print(percentual)
print("\nValor total para pessoas pcd")
print(soma_valor)
print("\nNúmero de vagas")
print(soma_vaga)
print("==========================================")
# Cotas pessoas pcd - Capitais
mask_cotas_pcd   = df_capitais['flag_cotas_pcd'].eq(1)
qtd_cotas_pcd    = mask_cotas_pcd.sum()
qtd_total           = len(df_capitais)
percentual          = qtd_cotas_pcd / qtd_total * 100 # 89.79%
soma_valor          = df_capitais['valor_cotas_pcd'].sum() # 9.942.526,06
soma_vaga           = df_capitais['vagas_cotas_pcd'].sum() 
print("Capitais")
print("==========================================")
print("% Cumprimento Cotas pcd")
print(percentual)
print("\nValor total para pessoas pcd")
print(soma_valor)
print("\nNúmero de vagas")
print(soma_vaga)
print("==========================================")


Geral
% Cumprimento Cotas
88.95582329317268

Valor total
78768581.61812568

Número de vagas
1591.0
Estados
% Cumprimento Cotas pcd
88.6039886039886

Valor total para pessoas pcd
68068705.82625924

Número de vagas
1270.0
Capitais
% Cumprimento Cotas pcd
89.79591836734694

Valor total para pessoas pcd
10699875.79186644

Número de vagas
321.0
